# Module 8: MIFID2_ext Staging (Non-PII)

This notebook creates the MIFID2_ext staging tables that do NOT require PII access.

**Replicates SSIS package:** `MIFID2.dtsx` (truncate/reload)

**Targets created:**
- `bi_output_regtechops_mifid2_ext_positionchangelog` — Position change log (ChangeTypeID=0 filter)
- `bi_output_regtechops_mifid2_ext_mirror` — Mirror operations with CopyFund derivation
- `bi_output_regtechops_mifid2_ext_hedgeexecutionlog` — Hedge execution log for report date
- `bi_output_regtechops_reg_liquidtyacount_scd` — SCD seed (already created)

**Blocked (PII dependency):** MIFID2_ext_Customer, MIFID2_ext_RegChange_Customer, MIFID2_ext_Position (depends on customer scope), MIFID2_ext_RegChange_Position, MIFID2_Failed_TRAX

In [0]:
dbutils.widgets.text("report_date", "2026-06-09", "Report Date (YYYY-MM-DD)")
report_date = dbutils.widgets.get("report_date")
print(f"Running Module 8: MIFID2_ext Non-PII Staging for report_date = {report_date}")

In [0]:
%sql
-- MIFID2_ext_PositionChangeLog: Position change log for report-date window
-- SSIS parity: History.PositionChangeLog day window with ChangeTypeID = 0
-- Source: main.trading.bronze_etoro_history_positionchangelog

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_positionchangelog
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_positionchangelog'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
)
SELECT
  pcl.PositionID,
  pcl.LastOpPriceRate AS ChangeLogLastOpPriceRate,
  pcl.Occurred AS ChangeLogOccurred,
  pcl.ChangeTypeID,
  pcl.IsSettled
FROM main.trading.bronze_etoro_history_positionchangelog pcl
JOIN run_window w
  ON pcl.Occurred >= w.start_ts
 AND pcl.Occurred < w.end_ts
WHERE pcl.ChangeTypeID = 0

In [0]:
%sql
-- MIFID2_ext_Mirror: Mirror operations with CopyFund derivation
-- SSIS parity: History.Mirror with MirrorOperationID = 1 and day window
-- CopyFund = 1 when parent CID has AccountTypeID = 9 in BackOffice source

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_mirror'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
),
backoffice_asof AS (
  SELECT CID, AccountTypeID
  FROM (
    SELECT
      b.CID,
      b.AccountTypeID,
      ROW_NUMBER() OVER (PARTITION BY b.CID ORDER BY b.ValidFrom DESC) AS rn
    FROM main.general.bronze_etoro_history_backofficecustomer b
    JOIN run_window w
      ON b.ValidFrom < w.end_ts
     AND b.ValidTo >= w.end_ts
  )
  WHERE rn = 1
)
SELECT
  m.MirrorID,
  m.ParentCID,
  m.MirrorOperationID,
  m.Occurred,
  CASE WHEN bo.AccountTypeID = 9 THEN 1 ELSE 0 END AS CopyFund
FROM main.trading.bronze_etoro_history_mirror m
JOIN run_window w
  ON m.Occurred >= w.start_ts
 AND m.Occurred < w.end_ts
LEFT JOIN backoffice_asof bo
  ON m.ParentCID = bo.CID
WHERE m.MirrorOperationID = 1

In [0]:
%sql
-- MIFID2_ext_HedgeExecutionLog: Hedge execution log for MIFID2 report
-- SSIS parity: ExecutionTime window, exclude (ProviderExecID IS NULL AND OrderState = 4)
-- Note: This is the MIFID2-specific subset with fewer columns than Reg_Ext_HedgeExecutionLog

CREATE OR REPLACE TABLE main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog
USING DELTA
LOCATION 'abfss://analysis@stgdpdlwe.dfs.core.windows.net/BI_OUTPUT/RegTechOps/mifid2_ext_hedgeexecutionlog'
AS
WITH run_window AS (
  SELECT
    CAST('${report_date}' AS TIMESTAMP) AS start_ts,
    CAST(date_add(CAST('${report_date}' AS DATE), 1) AS TIMESTAMP) AS end_ts
)
SELECT
  h.OrderID,
  h.HedgeServerID,
  h.InstrumentID,
  h.IsBuy,
  h.Units,
  h.ExecutionRate,
  h.ProviderExecID,
  h.ExecutionTime,
  h.Success,
  h.LogTime,
  h.LiquidityAccountID,
  h.EMSOrderID
FROM main.dealing.bronze_etoro_hedge_executionlog h
JOIN run_window w
  ON h.ExecutionTime >= w.start_ts
 AND h.ExecutionTime < w.end_ts
WHERE NOT (h.ProviderExecID IS NULL AND h.OrderState = 4)

In [0]:
%sql
-- Validation: row counts for Module 8 non-PII tables
SELECT 'mifid2_ext_positionchangelog' AS table_name, COUNT(*) AS row_count FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_positionchangelog
UNION ALL SELECT 'mifid2_ext_mirror', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_mirror
UNION ALL SELECT 'mifid2_ext_hedgeexecutionlog', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_mifid2_ext_hedgeexecutionlog
UNION ALL SELECT 'reg_liquidtyacount_scd', COUNT(*) FROM main.regtech_ops_stg.bi_output_regtechops_reg_liquidtyacount_scd